# Classificador Vizinho Mais Próximo (1-NN)

*Implementação em numpy · base original (sem pré-processamento)*

Cada amostra de teste recebe a classe da amostra de treino mais próxima segundo a distância de Minkowski de ordem $m$:

$$d_m(\mathbf{x}, \mathbf{z}) = \Big(\sum_{k} |x_k - z_k|^m\Big)^{1/m}, \qquad m \in \{1/2,\ 2/3,\ 1,\ 3/2,\ 2,\ 5/2\}$$

Para $m < 1$, $d_m$ não satisfaz a desigualdade triangular (não é métrica), mas continua definindo um critério de proximidade.

Protocolo (OBS 1–4 do enunciado): 100 rodadas treino-teste independentes e estratificadas para cada separação 20/80, 30/70, 50/50, 70/30 e 80/20.

## Dados

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # antes do numpy: evita falta de memória do OpenBLAS no Windows

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from avaliacao import (SPLITS, carregar, avaliar, estatisticas, matrizes_confusao, grafico_splits,
                       por_classe, grafico_por_classe, resumo, salvar)

pd.set_option("display.max_rows", 200)

X, y, classes = carregar(processada=False)
print(f"{len(y)} instâncias | {X.shape[1]} atributos | {len(classes)} classes: {list(classes)}")

5456 instâncias | 4 atributos | 4 classes: ['Move-Forward', 'Sharp-Right-Turn', 'Slight-Left-Turn', 'Slight-Right-Turn']


## Modelo

In [2]:
class VizinhoMaisProximo:
    """1-NN com distância de Minkowski de ordem m."""

    def __init__(self, m):
        self.m = m

    def fit(self, X, y):
        self.X_, self.y_ = X, y  # o 1-NN apenas memoriza o conjunto de treino
        return self

    def predict(self, X):
        # soma de |x_k - z_k|^m; a raiz 1/m é omitida por ser monotônica (não altera o vizinho mais próximo)
        d = sum(np.abs(X[:, [k]] - self.X_[:, k]) ** self.m for k in range(X.shape[1]))
        return self.y_[d.argmin(axis=1)]


ORDENS = {"1/2": 1/2, "2/3": 2/3, "1": 1, "3/2": 3/2, "2": 2, "5/2": 5/2}
MODELOS = {f"m = {nome}": VizinhoMaisProximo(m) for nome, m in ORDENS.items()}

## Execução
100 rodadas × 5 separações. Em cada rodada, todos os modelos usam a mesma divisão treino/teste.

In [ ]:
res, confusoes = avaliar(MODELOS, X, y, classes)
salvar(res, "classification", "vizinho_mais_prox")

## OBS 1 e 2 — Estatísticas de acurácia e tempos de treinamento/teste (ms)

In [ ]:
estatisticas(res).round(4)

## OBS 2 — Matrizes de confusão da melhor e da pior rodada

In [ ]:
matrizes_confusao(res, confusoes, classes, splits=["80/20"])

## OBS 3 — Efeito da separação treino/teste

In [ ]:
grafico_splits(res)

## OBS 4 — Desempenho por classe (taxa de acerto de cada classe)

In [ ]:
por_classe(res, classes).round(4)

In [ ]:
grafico_por_classe(res, classes)

## Resumo dos resultados

In [ ]:
resumo(res, classes)